In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q -U transformers accelerate peft bitsandbytes qwen-vl-utils datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 94.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 57.9 MB/s eta 0:00:00:00:0100:01


In [3]:
from datasets import load_dataset

sample = load_dataset("Rowan/vcr", "image_examples", split="validation", streaming=True)
row = next(iter(sample))

print("Top-level row keys:", row.keys())
print()
print("Number of questions attached to this image:", len(row["annotations"]))
print()
print("First annotation's keys:", row["annotations"][0].keys())
print()
print("First annotation, full contents:")
print(row["annotations"][0])

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Top-level row keys: dict_keys(['image', 'img_fn', 'metadata_fn', 'movie', 'objects', 'width', 'height', 'image_width', 'image_height', 'boxes', 'segms_json', 'question_count', 'source_zip_crc_mismatch', 'image_metadata_dimension_mismatch', 'annotations'])

Number of questions attached to this image: 4

First annotation's keys: dict_keys(['movie', 'objects', 'interesting_scores', 'answer_likelihood', 'img_fn', 'metadata_fn', 'answer_orig', 'question_orig', 'rationale_orig', 'question_tokens', 'answer_choice_tokens', 'answer_label', 'answer_match_iter', 'answer_sources', 'rationale_choice_tokens', 'rationale_sources', 'rationale_match_iter', 'rationale_label', 'source_zip_crc_mismatch', 'img_id', 'question_number', 'annot_id', 'match_fold', 'match_index', 'question_text', 'answer_choice_texts', 'rationale_choice_texts'])

First annotation, full contents:
{'movie': '1054_Harry_Potter_and_the_prisoner_of_azkaban', 'objects': ['person', 'person', 'person', 'car', 'cellphone', 'clock'], 'int

In [7]:
"""
vcr_data.py  (v4 -- Using pre-parsed text fields)
----------------------------------------------------
Loads VCR directly from https://huggingface.co/datasets/Rowan/vcr with
streaming=True. 
"""

import itertools
import re
from datasets import load_dataset
from PIL import ImageDraw

BOX_COLORS = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45",
]

# FIXED: Using the pre-parsed _text fields provided by Hugging Face
ANNOTATION_KEYS = {
    "question": "question_text",
    "answers": "answer_choice_texts",
    "rationales": "rationale_choice_texts",
    "answer_label": "answer_label",
    "rationale_label": "rationale_label",
}


def load_vcr_stream(split="validation"):
    ds = load_dataset("Rowan/vcr", "image_examples", split=split, streaming=True)
    return iter(ds)


def take_n(stream, n):
    return list(itertools.islice(stream, n))


def _get(ann, logical_key):
    real_key = ANNOTATION_KEYS[logical_key]
    if real_key not in ann:
        raise KeyError(
            f"Expected field '{real_key}' (for '{logical_key}') not found in this "
            f"annotation. Actual keys present: {list(ann.keys())}. "
        )
    return ann[real_key]


def replace_objects(text, objects):
    """
    Replaces [0], [1] etc. with [person1], [chair2] if they exist in the text.
    """
    if not isinstance(text, str):
        return text
        
    def repl(m):
        idx = int(m.group(1))
        if idx < len(objects):
            cls = objects[idx]
            return f"[{cls}{idx + 1}]"
        return m.group(0)
        
    return re.sub(r"\[(\d+)\]", repl, text)


def iter_questions(image_row):
    objects = image_row["objects"]
    annotations = image_row.get("annotations") or []

    for ann in annotations:
        yield {
            "question": replace_objects(_get(ann, "question"), objects),
            "answers": [replace_objects(a, objects) for a in _get(ann, "answers")],
            "rationales": [replace_objects(r, objects) for r in _get(ann, "rationales")],
            "answer_label": _get(ann, "answer_label"),
            "rationale_label": _get(ann, "rationale_label"),
        }


def draw_boxes(image, boxes, objects, indices=None):
    img = image.convert("RGB").copy()
    if indices is None:
        indices = range(len(boxes))

    draw = ImageDraw.Draw(img)
    for idx in indices:
        if idx >= len(boxes):
            continue
        x1, y1, x2, y2 = boxes[idx][:4]
        color = BOX_COLORS[idx % len(BOX_COLORS)]
        label = f"{objects[idx] if idx < len(objects) else 'obj'}{idx}"
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        text_w = 8 * len(label) + 6
        draw.rectangle([x1, max(0, y1 - 16), x1 + text_w, y1], fill=color)
        draw.text((x1 + 3, max(0, y1 - 15)), label, fill="white")
    return img

In [23]:
"""
prompts.py (v3 -- High Accuracy & Strict Formatting)
"""
import re

LETTERS = ["A", "B", "C", "D"]

def build_qa_prompt(question, answers):
    options = "\n".join(f"{LETTERS[i]}) {a}" for i, a in enumerate(answers))
    return (
        "You are an expert at Visual Commonsense Reasoning. "
        "The image has labeled people/objects in colored boxes (e.g., [person0]). "
        "Match the text to the visual boxes.\n\n"
        f"Question: {question}\n\n"
        f"Options:\n{options}\n\n"
        "Analyze the visual evidence, human intent, and social context in 1-2 sentences. "
        "Then, output the correct letter on the final line.\n"
        "Format:\nReasoning: <your brief reasoning>\nAnswer: <single letter A, B, C, or D>"
    )

def build_qar_prompt(question, chosen_answer, rationales):
    options = "\n".join(f"{LETTERS[i]}) {r}" for i, r in enumerate(rationales))
    return (
        "You are an expert at explaining WHY a visual commonsense answer is correct.\n\n"
        f"Question: {question}\n"
        f"Chosen answer: {chosen_answer}\n\n"
        f"Rationale options:\n{options}\n\n"
        "Select the rationale that best logically explains the chosen answer based on the image. "
        "Do not rewrite the options. Output the correct letter on the final line.\n"
        "Format:\nReasoning: <your brief reasoning>\nAnswer: <single letter A, B, C, or D>"
    )

def parse_letter(model_output):
    """
    Bulletproof parser. Looks for 'Answer: X', then 'answer is X', 
    then falls back to the very last standalone A, B, C, or D.
    """
    if not model_output:
        return None
        
    text = model_output.strip().upper()
    
    # Look for "Answer: X"
    m = re.search(r"ANSWER:\s*([ABCD])", text)
    if m:
        return m.group(1)
        
    # Look for "answer is X"
    m = re.search(r"ANSWER IS\s*([ABCD])", text)
    if m:
        return m.group(1)
        
    # Fallback: Find the very last standalone letter A, B, C, or D
    matches = re.findall(r"\b([ABCD])\b", text)
    if matches:
        return matches[-1]
        
    return None

In [9]:
"""
model_utils.py  (v2 -- Memory Optimized)
"""
import torch
import gc
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

class VCRModel:
    def __init__(self, model_name=MODEL_NAME, device="cuda", load_in_4bit=True):
        quant_kwargs = {}
        if load_in_4bit:
            from transformers import BitsAndBytesConfig
            quant_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
            )

        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            **quant_kwargs,
        )
        
        # FIX: Added max_pixels to cap image resolution and prevent OOM errors
        self.processor = AutoProcessor.from_pretrained(
            model_name, 
            max_pixels=1003520 
        )
        self.device = device

    @torch.no_grad()
    def generate(self, image, prompt, max_new_tokens=256):
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            }
        ]

        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(self.model.device)

        generated_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = self.processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        
        # FIX: Clear VRAM cache to prevent memory fragmentation over time
        del inputs, generated_ids, generated_ids_trimmed
        gc.collect()
        torch.cuda.empty_cache()
        
        return output_text[0]

In [10]:
model = VCRModel()

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

chat_template.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
stream = load_vcr_stream(split="validation")
sample_images = take_n(stream, 3)   # just 3 images to start
print(f"Pulled {len(sample_images)} images")
print("Questions on first image:", len(sample_images[0]["annotations"]))

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Pulled 3 images
Questions on first image: 4


In [24]:
image_row = sample_images[0]
image, boxes, objects = image_row["image"], image_row["boxes"], image_row["objects"]
annotated_image = draw_boxes(image, boxes, objects)

questions = list(iter_questions(image_row))
q = questions[0]

print("QUESTION:", q["question"])
print("ANSWERS:", q["answers"])

# Stage 1: Q -> A
qa_prompt = build_qa_prompt(q["question"], q["answers"])
qa_raw = model.generate(annotated_image, qa_prompt)
print("\nMODEL (answer stage) SAID:\n", qa_raw)

pred_answer_letter = parse_letter(qa_raw)
pred_answer_idx = LETTERS.index(pred_answer_letter)
print("\nPredicted answer:", pred_answer_letter, "| Correct answer:", LETTERS[q["answer_label"]])

# Stage 2: QA -> R (using the model's OWN chosen answer, per the task spec)
qar_prompt = build_qar_prompt(q["question"], q["answers"][pred_answer_idx], q["rationales"])
qar_raw = model.generate(annotated_image, qar_prompt)
print("\nMODEL (rationale stage) SAID:\n", qar_raw)

pred_rationale_letter = parse_letter(qar_raw)
print("\nPredicted rationale:", pred_rationale_letter, "| Correct rationale:", LETTERS[q["rationale_label"]])

QUESTION: How is [person0] feeling?
ANSWERS: ['[person0] is feeling amused.', '[person0] is upset and disgusted.', '[person0] is feeling very scared.', '[person0] is feeling uncomfortable with [person2].']

MODEL (answer stage) SAID:
 Reasoning: [person0] appears to be looking down and away from [person2], who seems to be laughing. This suggests that [person0] might be feeling uncomfortable or distressed by [person2]'s behavior. There is no clear indication of amusement or fear from [person0]'s body language.

Answer: D

Predicted answer: D | Correct answer: B

MODEL (rationale stage) SAID:
 Reasoning: The person labeled as [person0] has a twisted expression which suggests discomfort or disgust. This aligns with the chosen answer stating that [person0] is feeling uncomfortable with [person2].

Answer: D

Predicted rationale: D | Correct rationale: D


In [26]:
from tqdm import tqdm

stream = load_vcr_stream(split="validation")
N_QUESTIONS = 50  # start small, raise later

n_qa_correct = 0
n_qar_correct = 0
n_joint_correct = 0
n_total = 0

pbar = tqdm(total=N_QUESTIONS, desc="Evaluating")
for image_row in stream:
    if n_total >= N_QUESTIONS:
        break
    image, boxes, objects = image_row["image"], image_row["boxes"], image_row["objects"]
    annotated_image = draw_boxes(image, boxes, objects)
    for q in iter_questions(image_row):
        if n_total >= N_QUESTIONS:
            break
        qa_prompt = build_qa_prompt(q["question"], q["answers"])
        pred_a = parse_letter(model.generate(annotated_image, qa_prompt)) or "A"
        pred_a_idx = LETTERS.index(pred_a)
        answer_correct = pred_a_idx == q["answer_label"]

        qar_prompt = build_qar_prompt(q["question"], q["answers"][pred_a_idx], q["rationales"])
        pred_r = parse_letter(model.generate(annotated_image, qar_prompt)) or "A"
        pred_r_idx = LETTERS.index(pred_r)
        rationale_correct = pred_r_idx == q["rationale_label"]

        if answer_correct: n_qa_correct += 1
        if rationale_correct: n_qar_correct += 1
        if answer_correct and rationale_correct: n_joint_correct += 1
        n_total += 1
        pbar.update(1)
pbar.close()

print(f"N questions:     {n_total}")
print(f"Q->A accuracy:   {n_qa_correct/n_total:.2%}")
print(f"QA->R accuracy:  {n_qar_correct/n_total:.2%}")
print(f"Q->AR accuracy:  {n_joint_correct/n_total:.2%}")

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 50/50 [36:21<00:00, 43.63s/it] 

N questions:     50
Q->A accuracy:   68.00%
QA->R accuracy:  56.00%
Q->AR accuracy:  42.00%
